In [ ]:
 #test_prior.jl
# import other modules
include("readHist.jl")
include("PhysicalConstants.jl")
include("Lineshape.jl")
include("Dataset.jl")

using .LineshapeModule
using .DatasetModule
using .PhysicalConstants
using .readHistModule

In [ ]:
using Distributions, IntervalSets
using OrderedCollections

In [ ]:
LS1 = LineshapeGauss3("/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape/calibration/PeakShape_ds3819.root", 3819)
set_poly_scaling(LS1, kQvalue, "/Users/zhaokangkang/gssiwork/julia-dev/code-test/test_json/lineshape_scaling_ds3021.json",2)
set_poly_scaling(LS1, kSigma, "/Users/zhaokangkang/gssiwork/julia-dev/code-test/test_json/lineshape_scaling_ds3021.json",2)

In [ ]:
DS1 = DatasetModule.Dataset(LS1,
    3819,
    "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/super_reduced/background/unblinded/SuperReduced_Background_ds3819.root",
    0.947,
    0.0075,
    0.88345,
    0.00085,
    237.0,
    1.0,
    0.0,
    1.0,
    0.0,
    2465.0,
    2575.0,
    true,
    false,
    false,
    "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/CombinedEfficiency/CombinedEfficiencies.root",
    "",
    "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/Exposures/Exposures_ds3819.txt",
    "",
    true,
    "/Users/zhaokangkang/gssiwork/julia-dev/data/taup23_2ds/lineshape_scaling_output/CombinedReso.root")

In [ ]:
datasets = Vector{DatasetModule.Dataset}()
# using push! function for individual elements or append! for multiple elements
push!(datasets, DS1)

In [ ]:
using Plots
plot(DS1.bias_ls_scaling)
plot!(DS1.reso_ls_scaling)

In [ ]:
# sum over all groups' ds-ch of Effi*Exposure
sensExposure::Float64 = 0.0;
totExposure::Float64 = 0.0;
num = 0
for dataset in datasets
    for ch in (1:1:988)
        # sensExposure += dataset.total_efficiency_channel[ch] * dataset.exposure_channel[ch]
        # totExposure += dataset.exposure_channel[ch]
        eff::Float64 = get(dataset.total_efficiency_channel,ch, 0)    
        expo::Float64 = get(dataset.exposure_channel,ch, 0) 
        if (eff==0.0 || expo == 0.0)
            # println(ch)
            num += 1
        end
        sensExposure += expo*eff
        totExposure += expo
    end 
end
println(totExposure)
println(sensExposure)
println(num)

In [ ]:
# compute f_norm
# Abundance of 130Te is considered constant here
# i.e., included in fNorm directly
# if it is to be floated, then it has to be included
f_norm = PhysicalConstants.N_A * 1000. * PhysicalConstants.Abundance_130Te * sensExposure / PhysicalConstants.mass_TeO2


function signal_prior(datasets::Vector{DatasetModule.Dataset}, f_norm::Float64, bkg_only::Bool)
    nSgnCandidates::Float64 = 0.
    nBkgCandidates::Float64 = 0.
    for dataset in datasets
        for ev in dataset.events
            eventEnergy = ev.energy[1]
            if eventEnergy > PhysicalConstants.Emin0nbbPeak20keV &&
               eventEnergy < PhysicalConstants.Emax0nbbPeak20keV
                nSgnCandidates += 1.
            else
                nBkgCandidates += 1.
            end
        end
    end

    # width in keV of signal region
    sgnRange = PhysicalConstants.Emax0nbbPeak20keV - PhysicalConstants.Emin0nbbPeak20keV
    # Emax and Emin can be put outside if all datasets have the same
    fitRange = datasets[1].emax - datasets[1].emin  # assuming all datasets have the same fit range
    
    errNSgnCandidates = sqrt( nSgnCandidates + nBkgCandidates * sgnRange / (fitRange-sgnRange) )
    nSgnCandidates -= nBkgCandidates * sgnRange / (fitRange-sgnRange)
    
    minR = nSgnCandidates - 15.0 * errNSgnCandidates
    if minR < 0.0
        minR = 0.0
    end
    maxR = nSgnCandidates + 15.0 * errNSgnCandidates
    if maxR < 3.0
        maxR = 15.0
    end
    minR /= f_norm
    maxR /= f_norm

    distrS = minR .. maxR

    return distrS
end

sig_p = signal_prior(datasets, f_norm, false)

In [ ]:
#------------------------------------------------
# Add BI cts/kev/kg/yr parameters, shared among all datasets
#****************  B I  ***************************************
function BI_prior(datasets::Vector{DatasetModule.Dataset})
    nEvents::Float64 = 0.
    for dataset in datasets
        for ev in dataset.events
            eventEnergy = ev.energy[1]
            if eventEnergy < PhysicalConstants.Emin0nbbPeak40keV ||
                eventEnergy > PhysicalConstants.Emax0nbbPeak40keV
                nEvents += 1.
            end
        end
    end
    # width in keV of signal region
    exclusionRange = PhysicalConstants.Emax0nbbPeak40keV - PhysicalConstants.Emin0nbbPeak40keV
    fitRange = datasets[1].emax - datasets[1].emin  # assuming all datasets have the same fit range
    # # of bkg events = N_called outside 4 sigma region around Qbb times Fraction(= fitRange / (fitRange - exclusionRange))
    nEvents *= fitRange / (fitRange - exclusionRange)
    # HARD-CODED AVERAGE VALUE FOR TOTAL DETECTOR EFFICIENCY!!!
    nEvents *= 0.95; 
    # err_bkg_events = sqrt( N_called ) * fitRange / (fitRange - exclusionRange )
    errNEvents = sqrt( nEvents * fitRange / (fitRange - exclusionRange) )
    
    minNEvents = nEvents - 15. * errNEvents
    if minNEvents < 0.0 
        minNEvents = 0.0
    end
    maxNEvents = nEvents + 15. * errNEvents
    
    minBI = minNEvents / (fitRange*totExposure)
    maxBI = maxNEvents / (fitRange*totExposure)
    distrBI = minBI .. maxBI
    return distrBI
end
# end of BI prior
#*******************************************************  E N D   O F   B I

In [ ]:
BI_prior(datasets)

In [ ]:
#------------------------------------------------
# Add Linear Bkg parameters, shared among all datasets
#********* L I N E A R   B K G ********************************************** 
function BISlope_prior(datasets::Vector{DatasetModule.Dataset})
    Emax::Float64 = 0.0
    Emin::Float64 = 0.0
    Index::Int32 = 0
    for dataset in datasets
        if Index == 0
            Emax = dataset.emax
            Emin = dataset.emin
            Index += 1
            continue
        end
        if dataset.emax > Emax
            Emax = dataset.emax
        end
        if dataset.emin > Emin
            Emin = dataset.emin
        end
        Index += 1
    end
    # fMidRange = 0.5 * ( datasets[1].Emax + datasets[1].Emin )
    dE = 0.5 * (Emax - Emin)
    minSlope = -1.0 / dE
    maxSlope = 1.0 / dE
    distrBISlope = minSlope .. maxSlope
    return distrBISlope
end

# end of Linear Bkg prior
#********* E N D   O F   L I N E A R   B K G ********************************************** 

In [ ]:
BISlope_prior(datasets)

In [ ]:
#------------------------------------------------
# Add Linear Bkg parameters, shared among all datasets
#********* L I N E A R   B K G ********************************************** 
function Co60_prior(datasets::Vector{DatasetModule.Dataset})
#set one 60Co parameter shared
    # Floating 60Co mean value FIXME--hardcoded

    A::Float64 = 0.0 # [60Co events / kg / yr]
    errA::Float64 = 0.0
    B::Float64 = 0.0 # [bkg  events / kg / yr]
    errB::Float64 = 0.0
    totExp::Float64 = 0.0

    minCoW::Float64   = 2497.6
    maxCoW::Float64   = 2517.6
    max0nbbW::Float64 = 2537.6


    # Loop over datasets
    for dataset in datasets
        cutEff::Float64 = dataset.cut_efficiency
        CoReduction::Float64 = exp(-dataset.delta_t / PhysicalConstants.Tau60Cobalt)
        exposure::Float64 = 0.0 
        N60Co::Float64  = 0.0
        Nbkg::Float64   = 0.0
        energy::Float64 = 0.0
        # Loop over all channels
        exposure += DatasetModule.get_total_exposure(dataset)
        for ch in (1:1:988)
            for ev in dataset.events #dataset.events_channel[ch]
                energy = ev.energy[1]
                if energy > minCoW && energy < maxCoW
                    N60Co += 1.0
                end
                if energy < minCoW || energy > max0nbbW
                    Nbkg += 1.0
                end
            end# End of loop over events
        end# End of loop over channels
        A    +=      N60Co  / cutEff / CoReduction
        errA += sqrt(N60Co) / cutEff / CoReduction
        B    +=      Nbkg   / cutEff / CoReduction
        errB += sqrt(Nbkg)  / cutEff / CoReduction
        totExp += exposure

    end# End of loop over datasets

    A /= totExp
    errA /= totExp
    B /= totExp
    errB /= totExp

    dE = maxCoW - minCoW
    fitRange = datasets[1].emax - datasets[1].emin
    DE = fitRange - ( max0nbbW - minCoW )
    R = A - B * dE / DE
    errR = errA^2 + (errB * dE / DE)^2
    minR = R - 10. * errR
    maxR = R + 10. * errR

    if( minR < 0. )
        minR = 0.
    end
        
    distrCo60 = minR .. maxR
    return distrCo60

end

In [ ]:
Co60_prior(datasets)

In [ ]:
function Co60Mean_prior()
    distrCo60Mean = 2497. .. 2517.
    return distrCo60Mean

end
Co60Mean_prior()

In [ ]:
using EmpiricalDistributions
using Statistics
using StatsBase: Histogram
# --------------------------------------------------------------
# Add a nuisance parameter for the cut efficiency, if required

function cutEff_prior(datasets::Vector{DatasetModule.Dataset})
    Eff_priors = OrderedDict()
    
    # Check if efficiency uncertainty is provided
    flag = true

    # Loop over datasets
    for ds in datasets
        if ds.cut_efficiency_err == 0.0
            flag = false
        end
    end

    if !flag
        for ds in datasets
            if !ds.th1_efficiency
                error("Efficiency set as nuisance parameter but its uncertainty is not provided. Abort.")
            end
        end
    end

    # Loop over datasets
    for ds in datasets

        # --------------------------------------------------
        # Case 1: Gaussian efficiency prior
        if !ds.th1_efficiency

            eff    = ds.cut_efficiency
            effErr = ds.cut_efficiency_err

            minEff = eff - 5 * effErr
            maxEff = eff + 5 * effErr

            minEff = max(minEff, 0.0)
            maxEff = min(maxEff, 1.0)

            name      = "EffCut_ds$(ds.ds))"
            Eff_priors[Symbol(name)] = Truncated(Normal(eff, effErr), minEff, maxEff)
            
        # --------------------------------------------------
        # Case 2: Histogram-based efficiency prior
        else
            histo = truncate_histogram(ds.eff_prior,0.999)
            d_histo = UvBinnedDist(histo)
            name      = "EffCut_ds$(ds.ds))"
            Eff_priors[Symbol(name)] = d_histo
        end
    end
    return Eff_priors
    
end

In [ ]:
using Plots
# plot(DS1.eff_prior)
histo = DS1.eff_prior
d_histo2 = UvBinnedDist(histo)
x = range(0.75, 1.2; length=1000)
# PDF
y = pdf.(d_histo2, x)
# Plot
plot(x, y,
     xlabel="x",
     ylabel="PDF",
     label="Normal(0,1)",
     linewidth=2)

In [ ]:
priors = OrderedDict()
xmin = 0.5
xmax = 5.5
mean = 2.0
sigma = 1.0
priors["ds1"] = Truncated(Normal(mean, sigma), xmin, xmax)
dist = priors["ds1"]
# x range
x = range(xmin-2, xmax+2; length=1000)
# PDF
y = pdf.(dist, x)
# Plot
plot(x, y,
     xlabel="x",
     ylabel="PDF",
     label="Normal(0,1)",
     linewidth=2)

In [ ]:
Eff_priors = OrderedDict()
ds = DS1
eff    = ds.cut_efficiency
effErr = ds.cut_efficiency_err

minEff = eff - 5 * effErr
maxEff = eff + 5 * effErr
minEff = max(minEff, 0.0)
maxEff = min(maxEff, 1.0)

name      = "EffCut_ds$(ds.ds))"
Eff_priors[Symbol(name)] = Truncated(Normal(eff, effErr), minEff, maxEff)

In [ ]:
priors_tmp = cutEff_prior(datasets)
d_histo = priors_tmp[Symbol("EffCut_ds3819)")]
x = range(0.75, 1.2; length=1000)
# PDF
y = pdf.(d_histo, x)
# Plot
plot(x, y,
     xlabel="x",
     ylabel="PDF",
     label="Normal(0,1)",
     linewidth=2)

In [ ]:
# --------------------------------------------------------------
# Add a nuisance parameter for the Monte Carlo efficiency
# (common to all datasets)
function MCEff_prior(datasets::Vector{DatasetModule.Dataset})
    # Check if MC efficiency and uncertainty are provided and consistent
    flag = true
    for ds in datasets
        if ds.containment_efficiency != PhysicalConstants.MCEfficiency ||
           ds.containment_efficiency_err  != PhysicalConstants.MCEfficiencyErr ||
           ds.containment_efficiency_err  == 0.0
            flag = false
        end
    end
    if !flag
        error(
            "Monte Carlo efficiency set as nuisance parameter but in the current formulation (PRL19 analysis):\n" *
            "This is a parameter common to all datasets (both value and error).\n" *
            "This error can mean that:\n" *
            "Option 1: the MC efficiency value or the error differs from PhysicalConstants0nbb\n" *
            "Option 2: the uncertainty on the MC efficiency is not provided (null). Abort."
        )
    end
    minMCEff = PhysicalConstants.MCEfficiency - 5.0*PhysicalConstants.MCEfficiencyErr
    maxMCEff = PhysicalConstants.MCEfficiency + 5.0*PhysicalConstants.MCEfficiencyErr

    println(
        "This is the range for the MC efficiency (added as NP to the Model for the ML fit): ",
        minMCEff,
        " ",
        maxMCEff
    )
    return Truncated(Normal(PhysicalConstants.MCEfficiency, PhysicalConstants.MCEfficiencyErr), minMCEff, maxMCEff)
end
# ------------------------------------
# Add a parameter for Qbb
function Qββ_prior()
    minQββ = PhysicalConstants.Qββ130Te - 5.0*PhysicalConstants.Qββ130TeErr
    maxQββ = PhysicalConstants.Qββ130Te + 5.0*PhysicalConstants.Qββ130TeErr
    return Truncated(Normal(PhysicalConstants.Qββ130Te, PhysicalConstants.Qββ130TeErr), minQββ, maxQββ)
end
# ------------------------------------------------------
# Add a parameter for the isotopic fraction
function IsoFrac_prior()

    minAb = PhysicalConstants.NaturalIsotopicAbuTe130 - 5.0*PhysicalConstants.NaturalIsotopicAbuTe130Err
    maxAb = PhysicalConstants.NaturalIsotopicAbuTe130 + 5.0*PhysicalConstants.NaturalIsotopicAbuTe130Err
    
    return Truncated(Normal(PhysicalConstants.NaturalIsotopicAbuTe130, PhysicalConstants.NaturalIsotopicAbuTe130Err), minAb, maxAb)
   
end
# ------------------------------------------------------
# Add a parameter for the PSA efficiency systematic
function SystEff_prior() 
    @warn "You need to take care the prior building of systematical efficiency!"
    SystEffMin = 0.9
    SystEffMax = 1.2

    return SystEffMin .. SystEffMax
end

In [ ]:
pris = [MCEff_prior(datasets),Qββ_prior(),IsoFrac_prior(),SystEff_prior() ]
append!(pris, [SystEff_prior()])

In [ ]:
histo_bias = ds.bias_ls_scaling
length(histo_bias.weights)
length(histo_bias.edges[1])

function trim_histo_zerobins(hist::Histogram)
    bin_start::Int64 = 1
    bin_end::Int64 = length(hist.weights)
    println("Original bin range: start = '$bin_start', end = '$bin_end'.")
    for i in 1:1:length(hist.weights)
        if hist.weights[i]==0.0
            continue
        end
        bin_start = i
        break
    end
    for i in length(hist.weights):-1:1
        if hist.weights[i]==0.0
            continue
        end
        bin_end = i
        break
    end
    println("Trimmed bin range: start = '$bin_start', end = '$bin_end'.")
    trimmed_hist = Histogram(hist.edges[1][bin_start:bin_end+1],hist.weights[bin_start:bin_end])
    return trimmed_hist
end
trimmed_hist = trim_histo_zerobins(histo_bias)
plot(trimmed_hist)



In [ ]:
bin_x = [0.,1.,2.,3.,4.,5.,6.,7.,8.,9.,10.,11.,12.,13.,14.,15.,16.]
weight_y = [0.0,0,0,2,3,4,5,6,7,0,3,0,0.0,0.0,0.0,0]
test_hist = Histogram(bin_x, weight_y)
trim_histo_zerobins(test_hist)

In [ ]:
bin_x = [0.,1.,2.,3.,4.,5.,6.,7.,8.,9.,10.,11.,12.,13.,14.,15.,16.]
weight_y = [0.0,0,0,2,3,4,5,6,7,0,3,0,0.0,0.0,0.0,0]
test_hist = Histogram(bin_x, weight_y)
trimmed_hist = trim_histo_zerobins(test_hist)
plot(test_hist)
plot!(trimmed_hist)

In [ ]:
# --------------------------------------------------------------
using LinearAlgebra
# Lineshape scaling nuisance parameters
function LineshapeScaling_prior(datasets::Vector{DatasetModule.Dataset}, fHistoLineshape::Bool)

    @warn "Lineshape scaling still under development. Run at your own risk."
    # Loop over datasets
    bias_priors = OrderedDict()
    reso_priors = OrderedDict()
    Qval_priors = OrderedDict()
    Sigma_priors = OrderedDict()
    for ds in datasets

        # ======================================================
        # Case 1: histogram-based lineshape priors
        if fHistoLineshape

            # ---------------- Bias ----------------
            hist_bias = ds.bias_ls_scaling
            hist_bias = trim_histo_zerobins(hist_bias)
            hist_bias = truncate_histogram(hist_bias,0.999)
            bias_name = "bias_ds$(ds.ds))"
            bias_priors[Symbol(bias_name)] = UvBinnedDist(hist_bias)

            # ---------------- Resolution ----------------
            hist_reso = ds.reso_ls_scaling
            hist_reso = trim_histo_zerobins(hist_reso)
            hist_reso = truncate_histogram(hist_reso,0.999)
            reso_name = "reso_ds$(ds.ds))"
            reso_priors[Symbol(reso_name)] = UvBinnedDist(hist_reso)

        # ======================================================
        # Case 2: analytic lineshape with covariance matrices
        else
            ls = ds.lineshape
            # Q-value scaling
            pQval = ls.fPolyScalingMap[kQvalue]
            mQval = ls.fCovarianceMatrixPar[kQvalue]
            # Sigma scaling
            pSigma = ls.fPolyScalingMap[kSigma]
            mSigma = ls.fCovarianceMatrixPar[kSigma]

            mvQval = MvNormal(pQval, Symmetric(mQval))
            mvSigma = MvNormal(pSigma, Symmetric(mSigma))

            Qval_name = "Qval_ds$(ds.ds))"
            Qval_priors[Symbol(Qval_name)] = mvQval

            Sigma_name = "Sigma_ds$(ds.ds))"
            Sigma_priors[Symbol(Sigma_name)] = mvSigma
            
        end
    end
    return bias_priors, reso_priors, Qval_priors, Sigma_priors
end

In [ ]:
LineshapeScaling_prior(datasets, false)
LineshapeScaling_prior(datasets, true)

In [ ]:
using BAT
using Distributions
using Random
using StatsPlots
using ValueShapes
using LinearAlgebra
using Statistics
using DensityInterface
include("TruncatedMvNormal.jl")
function ParaScaling__prior(datasets::Vector{DatasetModule.Dataset})
    @warn "Lineshape parameters scaling still under development. Don't call this."
    Qval_priors = OrderedDict()
    Sigma_priors = OrderedDict()
    nSigma::Float64 = 5.0
    ls = ds.lineshape
    # Q-value scaling
    pQval = ls.fPolyScalingMap[kQvalue]
    mQval = ls.fCovarianceMatrixPar[kQvalue]

    diagQval = [ mQval[i,i] for i in 1:size(mQval)[1]]
    println(diagQval)
    lowerQval = pQval .- nSigma .* diagQval
    upperQval = pQval .+ nSigma .* diagQval
    println(lowerQval)
    println(upperQval)
    
    # Sigma scaling
    pSigma = ls.fPolyScalingMap[kSigma]
    mSigma = ls.fCovarianceMatrixPar[kSigma]
    diagSigma = [ mSigma[i,i] for i in 1:size(mSigma)[1]]
    println(diagSigma)
    lowerSigma = pSigma .- nSigma .* diagSigma
    upperSigma = pSigma .+ nSigma .* diagSigma
    println(lowerSigma)
    println(upperSigma)
    
    # mvQval = MvNormal(pQval, Symmetric(mQval))
    # mvSigma = MvNormal(pSigma, Symmetric(mSigma))
    mvQval = TruncatedMvNormal(pQval, Symmetric(mQval), lowerQval, upperQval)
    mvSigma = TruncatedMvNormal(pSigma, Symmetric(mSigma), lowerSigma, upperSigma)
    
    
    Qval_name = "Qval_ds$(ds.ds))"
    Qval_priors[Symbol(Qval_name)] = mvQval
    
    Sigma_name = "Sigma_ds$(ds.ds))"
    Sigma_priors[Symbol(Sigma_name)] = mvSigma
    @warn "This user-defined truncated MvNormal distribution can not be used as prior in BAT."
    @warn "Further work needs to be focused on."

    return Qval_priors, Sigma_priors
end

In [ ]:
# LineshapeScaling_prior(datasets,true)

In [ ]:
ParaScaling__prior(datasets)